# 10 — Validate Deployment

Run this notebook after `databricks bundle deploy` + `databricks bundle run data_mesh_orchestrator` to verify everything was recreated correctly.

**Checks:** Catalogs, schemas, tables, row counts, data quality, dashboard, Genie space, alert, service principal.

In [0]:
import sys

errors = []
checks_passed = 0

def check(description, condition, detail=''):
    global checks_passed
    if condition:
        checks_passed += 1
        print(f'  ✓ {description} {detail}')
    else:
        errors.append(f'{description} {detail}')
        print(f'  ✗ FAIL: {description} {detail}')

print('═' * 60)
print('  Data Mesh Demo — Post-Deployment Validation')
print('═' * 60)


In [0]:
# ── 1. Catalogs ──
print('\n1️⃣  Catalogs')
for cat in ['adoit_product', 'snow_product', 'data_mesh_hub']:
    try:
        spark.sql(f'DESCRIBE CATALOG {cat}').collect()
        check(cat, True)
    except:
        check(cat, False, '— catalog missing')

# ── 2. Schemas ──
print('\n2️⃣  Schemas')
expected_schemas = {
    'adoit_product': ['bronze', 'silver', 'gold'],
    'snow_product': ['bronze', 'silver', 'gold'],
    'data_mesh_hub': ['cross_domain', 'platform']
}
for cat, schemas in expected_schemas.items():
    for schema in schemas:
        try:
            spark.sql(f'DESCRIBE SCHEMA {cat}.{schema}').collect()
            check(f'{cat}.{schema}', True)
        except:
            check(f'{cat}.{schema}', False, '— schema missing')


In [0]:
# ── 3. Tables & Row Counts ──
print('\n3️⃣  Tables & Row Counts')
expected_tables = {
    'adoit_product.bronze.raw_applications': 10,
    'adoit_product.bronze.raw_business_capabilities': 10,
    'adoit_product.bronze.raw_app_capability_map': 13,
    'snow_product.bronze.raw_incidents': 12,
    'snow_product.bronze.raw_change_requests': 8,
    'adoit_product.silver.applications': 10,
    'adoit_product.silver.business_capabilities': 10,
    'snow_product.silver.incidents': 12,
    'snow_product.silver.change_requests': 8,
    'adoit_product.gold.application_landscape': 10,
    'snow_product.gold.service_health': 9,
    'data_mesh_hub.cross_domain.application_risk_profile': 10,
    'data_mesh_hub.platform.data_product_registry': 3,
    'data_mesh_hub.platform.data_contracts': 5,
    'data_mesh_hub.platform.domain_maturity_scorecard': 3,
}
for table, expected in expected_tables.items():
    try:
        count = spark.sql(f'SELECT COUNT(*) AS cnt FROM {table}').collect()[0]['cnt']
        check(table, count == expected, f'— {count} rows (expected {expected})')
    except Exception as e:
        check(table, False, f'— {str(e)[:60]}')


In [0]:
# ── 4. Gold Product Data Quality ──
print('\n4️⃣  Gold Data Product Quality')
try:
    al = spark.sql('SELECT COUNT(*) AS t, COUNT(app_name) AS n, COUNT(criticality) AS c FROM adoit_product.gold.application_landscape').collect()[0]
    check('Application Landscape completeness', al['n'] == al['t'] and al['c'] == al['t'], f'— {al["t"]} rows, 100%')
except Exception as e:
    check('Application Landscape', False, str(e)[:60])

try:
    rp = spark.sql('SELECT COUNT(*) AS t, COUNT(composite_risk_score) AS r FROM data_mesh_hub.cross_domain.application_risk_profile').collect()[0]
    check('Risk Profile completeness', rp['r'] == rp['t'], f'— {rp["t"]} rows')
except Exception as e:
    check('Risk Profile', False, str(e)[:60])

try:
    sh = spark.sql('SELECT COUNT(*) AS t, COUNT(risk_score) AS r FROM snow_product.gold.service_health').collect()[0]
    check('Service Health completeness', sh['r'] == sh['t'], f'— {sh["t"]} rows')
except Exception as e:
    check('Service Health', False, str(e)[:60])


In [0]:
# ── 5. Workspace Assets ──
print('\n5️⃣  Workspace Assets')
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

try:
    d = w.lakeview.get('01f1391892551310ac80ed4aed493170')
    check('Dashboard', True, f'— {d.display_name}')
except:
    check('Dashboard', False, '— not found')

try:
    a = w.alerts.get('8548d7a9-16d2-4a3b-b281-d037eb053798')
    check('SQL Alert', True, f'— {a.display_name}')
except:
    check('SQL Alert', False, '— not found')

try:
    sps = list(w.service_principals.list(filter="displayName eq 'data-mesh-cicd'"))
    check('Service Principal', len(sps) > 0, f'— {sps[0].application_id}' if sps else '— missing')
except:
    check('Service Principal', False, '— error')


In [0]:
# ── FINAL RESULTS ──
print(f'\n{"═" * 60}')
print(f'  RESULTS: {checks_passed} passed, {len(errors)} failed')
print(f'{"═" * 60}')

if errors:
    print('\n❌ FAILURES:')
    for e in errors:
        print(f'   - {e}')
    print('\n⚠️  Fix the above issues and re-run validation.')
else:
    print('\n✅ ALL CHECKS PASSED — Demo is fully operational!')
    print('\n   Safe to remove duplicate artifacts:')
    print('   - data_mesh_demo/ folder (originals)')
    print('   - New Notebook 2026-04-23 22:43:52 (scratch)')
